In [0]:
from pyspark.sql.functions import *

s = spark.table("default.silver_retail")

monthly = (s
    .groupBy("invoice_year", "invoice_month", "country")
    .agg(
        round(sum("revenue"), 2).alias("total_revenue"),
        countDistinct("invoice_no").alias("num_invoices"),
        countDistinct("customer_id").alias("unique_customers"),
        round(avg("unit_price"), 2).alias("avg_unit_price")
    )
    .orderBy("invoice_month", "country")
)

monthly.write.format("delta").mode("overwrite") \
    .saveAsTable("default.gold_monthly_revenue")

print(f"gold_monthly_revenue ✓  rows: {spark.table('default.gold_monthly_revenue').count():,}")

In [0]:
products = (s
    .groupBy("stock_code", "description")
    .agg(
        round(sum("revenue"), 2).alias("total_revenue"),
        sum("quantity").alias("units_sold"),
        countDistinct("invoice_no").alias("num_orders"),
        round(avg("unit_price"), 2).alias("avg_price")
    )
    .orderBy(desc("total_revenue"))
)

products.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.gold_top_products")

print(f"gold_top_products ✓  rows: {spark.table('default.gold_top_products').count():,}")

In [0]:
max_date = s.agg(max("invoice_date")).collect()[0][0]

rfm = (s
    .groupBy("customer_id", "country")
    .agg(
        datediff(lit(max_date), max("invoice_date")).alias("recency_days"),
        countDistinct("invoice_no").alias("frequency"),
        round(sum("revenue"), 2).alias("monetary_value")
    )
    .withColumn("segment",
        when(col("monetary_value") >= 5000, "High value")
        .when(col("monetary_value") >= 1000, "Mid value")
        .otherwise("Low value"))
)

rfm.write.format("delta").mode("overwrite") \
    .saveAsTable("default.gold_customer_rfm")

print(f"gold_customer_rfm ✓  rows: {spark.table('default.gold_customer_rfm').count():,}")

In [0]:
dow = (s
    .withColumn("day_name", date_format("invoice_date", "EEEE"))
    .groupBy("day_of_week", "day_name")
    .agg(
        round(sum("revenue"), 2).alias("total_revenue"),
        countDistinct("invoice_no").alias("num_invoices")
    )
    .orderBy("day_of_week")
)

dow.write.format("delta").mode("overwrite") \
    .saveAsTable("default.gold_dow_sales")

print(f"gold_dow_sales ✓  rows: {spark.table('default.gold_dow_sales').count():,}")

In [0]:
tables = [
    "gold_monthly_revenue",
    "gold_top_products",
    "gold_customer_rfm",
    "gold_dow_sales"
]

print("=== Gold layer summary ===")
for t in tables:
    count = spark.table(f"default.{t}").count()
    print(f"  default.{t}: {count:,} rows")

print("\n=== Sample: top 5 products ===")
spark.sql("""
    SELECT description, total_revenue, units_sold, num_orders
    FROM default.gold_top_products
    LIMIT 5
""").show(truncate=False)

In [0]:
# Confirm all 4 tables are queryable
for t in ["gold_monthly_revenue", "gold_top_products", 
          "gold_customer_rfm", "gold_dow_sales"]:
    spark.sql(f"SELECT 1 FROM default.{t} LIMIT 1").collect()
    print(f"default.{t} ✓")

print("\nAll Gold tables healthy — ready to build Job and Dashboard")

In [0]:
dbutils.notebook.exit("gold_tables=4")
